### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Traking agent behabior with logging, analytics and debugging
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")

#### Summarization Middleware
Automatically summarize conversation history when approching token limits, preversing recent messages while compressing older contenxt. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model = "gpt-4o-mini",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger= ("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [3]:
### Run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='c203a6a4-edc3-4ea1-9352-da796c8cf7e4'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7jnP5t4wLIKKdwGptzw87v9JNXBN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c4821-e32b-79f0-819a-1a82d016a5db-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

### Token Size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, bussiness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model = "gpt-4o-mini",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        ),
    ]
)
config = {"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [6]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Mexico City"]

for city in cities:
    response = agent.invoke(
        {"messages":  [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(f"{(response['messages'])}")

Paris: ~145 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='69c42580-0803-4c77-ab6e-aca3c3786dcc'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7jnbZ0jRL7akRJBB47n9qfX8KHlr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4822-14be-7b60-ae46-d4345998a506-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_23BhGD2zzstgkm39NGRPhqca', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53

### Fraction

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Gran Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing
agent = create_agent(
    model = "gpt-4o-mini",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware =[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger = ("fraction", 0.005), # 0.5% = ~640 tokens
            keep = ("fraction", 0.002),    # 0.2% = ~256 tokens
        )
    ]
)

config = {"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Mexico City"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config = config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response["messages"])} msgs")
    print(response["messages"])

Paris: ~64 tokens (0.0500%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='24655872-68d0-4e98-9b3c-9413f1b4bc11'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 44, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6c0d1490cb', 'id': 'chatcmpl-D7joGy2z5xxA92psvQmxn4oNDpBV5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4822-b475-7920-981a-47200af412e6-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_yDjeArrDZzKLzFmqjZwm0kKj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 44

### Humman in the Loop Middleware
Pause agent execution for human approval, editing, or rejection of too calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Complience workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject: '{subject}'"


In [9]:
agent = create_agent(
    model = "gpt-4o-mini",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email_tool": {
                        "allowed_decisions": ["approve", "edit", "reject"]
                    },
                    "read_email_tool": False
                }
            )
        ]
)

In [10]:
config = {"configurable" : {"thread_id" : "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [11]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='55e0fbde-77c0-4844-a238-cad16bf045a1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7joZr4Abtld5S9CL9wCpujEYHO5n', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4823-008c-7062-88ca-5d77210fd4fc-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_BQselgDJsVh

In [12]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
         resume = {
            "decisions": [
                {"type": "approve"}
            ]
         }   
        ), 
        config = config
    )
    print(f"Result: {result["messages"][-1].content}")

Paused! Approving...
Result: The email has been sent to john@test.com with the subject "Hello" and the body "How are you?".


In [13]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='55e0fbde-77c0-4844-a238-cad16bf045a1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7joZr4Abtld5S9CL9wCpujEYHO5n', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4823-008c-7062-88ca-5d77210fd4fc-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_BQselgDJsVh

### Reject

In [14]:
agent = create_agent(
    model = "gpt-4o-mini",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email_tool": {
                        "allowed_decisions": ["approve", "edit", "reject"]
                    },
                    "read_email_tool": False
                }
            )
        ]
)

In [15]:
config = {"configurable" : {"thread_id" : "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [16]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7abff2b6-5f8d-4b69-bdaf-a3370a3aa105'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7jobGC6CPmHRl9gNf8uer7zYaEvJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4823-0861-7200-9c68-8489aaad12a0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_AhVOhlfWlUc

In [17]:
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
         resume = {
            "decisions": [
                {"type": "reject"}
            ]
         }   
        ), 
        config = config
    )
    print(f"Result: {result["messages"][-1].content}")

Paused! Approving...
Result: It seems that I am currently unable to send the email. If you'd like, I can help you with drafting the email or providing instructions on how to send it yourself. Would you like assistance with that?


In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7abff2b6-5f8d-4b69-bdaf-a3370a3aa105'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7jobGC6CPmHRl9gNf8uer7zYaEvJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4823-0861-7200-9c68-8489aaad12a0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_AhVOhlfWlUc

### Editing

In [19]:
agent = create_agent(
    model = "gpt-4o-mini",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email_tool": {
                        "allowed_decisions": ["approve", "edit", "reject"]
                    },
                    "read_email_tool": False
                }
            )
        ]
)

In [20]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [21]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='77a9d097-eb42-423e-adfd-999a2bd02345'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 95, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7joe6MRXJ3BHTlI4gWkKxk7qhRr6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4823-113a-7ed2-b989-91c3a650cabf-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'subject': 'Test', 'body': 'Hello'}, 'id': 'call_uvvpEIgUXeDdeljIVmagRZC

In [22]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Editing...
Result: The email has been successfully sent to correct@email.com with the subject "Corrected Subject" and the body "This was edited by human before sending." 

If you need anything else, feel free to ask!


In [23]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='77a9d097-eb42-423e-adfd-999a2bd02345'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 95, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7joe6MRXJ3BHTlI4gWkKxk7qhRr6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4823-113a-7ed2-b989-91c3a650cabf-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was